# Test: Repo Restructure (`refactor/repo-structure`)

Verify that the restructured codebase imports and runs correctly on Colab Pro.

## 0. Setup

In [ ]:
# Clone the feature branch
!rm -rf Conformal-training
!git clone --branch refactor/repo-structure https://github.com/iemppu/Conformal-training.git
%cd Conformal-training
!git log --oneline -1

In [ ]:
# Install dependencies (jax needed by smooth_conformal)
!pip install -q jax jaxlib scikit-learn tqdm

In [ ]:
import sys, os
# Ensure repo root is on path
REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print(f"Repo root: {REPO_ROOT}")
print(f"Directory listing: {os.listdir('.')}")

## 1. Import Tests — Models

In [ ]:
from src.models.resnet import resnet
from src.models.vgg import vgg16, vgg19_bn
from src.models.densenet import densenet

print("resnet:", resnet)
print("vgg16:", vgg16)
print("densenet:", densenet)
print("[PASS] src.models imports OK")

## 2. Import Tests — Methods (no arg-parsing deps)

In [ ]:
from src.methods.losses import LDAMLoss, FocalLoss
print("LDAMLoss:", LDAMLoss)
print("FocalLoss:", FocalLoss)
print("[PASS] src.methods.losses OK")

In [ ]:
from src.methods import isotonic
print("isotonic:", isotonic)
print("[PASS] src.methods.isotonic OK")

In [ ]:
from src.methods import split_conformal
print("split_conformal:", split_conformal)
print("[PASS] src.methods.split_conformal OK")

In [ ]:
from src.methods.sorting_nets import comm_pattern_batcher
print("comm_pattern_batcher:", comm_pattern_batcher)
print("[PASS] src.methods.sorting_nets OK")

In [ ]:
from src.methods.variational_sorting_net import VariationalSortingNet
print("VariationalSortingNet:", VariationalSortingNet)
print("[PASS] src.methods.variational_sorting_net OK")

In [ ]:
from src.methods.smooth_conformal import smooth_aps_score, smooth_aps_score_all
print("smooth_aps_score:", smooth_aps_score)
print("[PASS] src.methods.smooth_conformal OK")

## 3. Import Tests — Methods (arg-parsing chain)

`pytorch_ops` → `numpy_ops` → `isotonic` and `pytorch_ops` also imports `config` (parser).

`scores` imports `pytorch_ops`, `smooth_conformal`, `metrics`, etc.

In [ ]:
from src.methods.numpy_ops import isotonic_l2, isotonic_kl
print("isotonic_l2:", isotonic_l2)
print("[PASS] src.methods.numpy_ops OK")

In [ ]:
# pytorch_ops triggers argparse at import time — this tests the full chain
from src.methods.pytorch_ops import soft_rank, soft_sort
print("soft_rank:", soft_rank)
print("soft_sort:", soft_sort)
print("[PASS] src.methods.pytorch_ops OK")

In [ ]:
from src.methods.scores import get_HPS_scores, compute_scores_diff, Smoothquantile
print("get_HPS_scores:", get_HPS_scores)
print("[PASS] src.methods.scores OK")

## 4. Import Tests — Data

In [ ]:
from src.data.cifar100 import load_cifar100
print("load_cifar100:", load_cifar100)
print("[PASS] src.data.cifar100 OK")

## 5. Import Tests — Utils

In [ ]:
from src.utils import config as parser_file
print("config.initialize:", parser_file.initialize)
print("[PASS] src.utils.config OK")

In [ ]:
from src.utils.metrics import evaluate_predictions, get_scores_HPS, get_scores, classwise_conformal, Marginal_conformal
print("evaluate_predictions:", evaluate_predictions)
print("[PASS] src.utils.metrics OK")

## 6. Import Tests — conformal_utils (the big one)

This file imports from models, methods, data, and utils — it's the integration test.

In [ ]:
from src.methods.conformal_utils import (
    Estimate_quantile_n, UniformMatchingLoss, PinballMarginal,
    load_train_objs, base_path_for_finetune, load_checkpoint,
    prepare_dataloader, loss_fnc, check_path, create_final_data,
    create_folder, test_model, loss_cal, create_optimizers,
    find_scores_APS, find_scores_HPS, find_scores_RAPS,
)
print("[PASS] src.methods.conformal_utils OK — all key symbols imported")

## 7. Smoke Test — Instantiate a model & run forward pass

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# ResNet-20 on CIFAR-100
model = resnet(depth=20, num_classes=100, use_fc_single=False).to(device)
x = torch.randn(4, 3, 32, 32).to(device)
out = model(x)
print(f"ResNet-20 output shape: {out.shape}")  # expect [4, 100]
assert out.shape == (4, 100)
print("[PASS] ResNet-20 forward pass OK")

In [ ]:
# DenseNet-100 on CIFAR-100
model_dn = densenet(depth=100, dropRate=0, num_classes=100, growthRate=12, compressionRate=2, use_fc_single=False).to(device)
out_dn = model_dn(x)
print(f"DenseNet-100 output shape: {out_dn.shape}")
assert out_dn.shape == (4, 100)
print("[PASS] DenseNet-100 forward pass OK")

## 8. Smoke Test — Load CIFAR-100 data

In [ ]:
train_loader, val_loader, test_loader_all, num_train, num_val, train_ds, val_ds, test_ds = \
    load_cifar100(save_path=None, n_tr=50, n_val=10, n_cal=10, n_test=10, train_rho=1.0, val_rho=1.0, num_classes=100)

print(f"Train dataset size: {len(train_ds)}")
print(f"Val dataset size:   {len(val_ds)}")
print(f"Test dataset size:  {len(test_ds)}")

# Grab one batch
imgs, labels = next(iter(train_loader))
print(f"Batch shape: {imgs.shape}, labels shape: {labels.shape}")
print("[PASS] CIFAR-100 data loading OK")

## 9. Smoke Test — Scoring functions

In [ ]:
model.eval()
with torch.no_grad():
    imgs_d = imgs.to(device)
    labels_d = labels.to(device)
    logits = model(imgs_d)
    probs = torch.nn.Softmax(dim=1)(logits)

    # HPS scores
    hps = get_HPS_scores(probs, labels_d)
    print(f"HPS scores shape: {hps.shape}, mean: {hps.mean():.4f}")

print("[PASS] Scoring functions OK")

## 10. Functional Tests — Scoring Functions (HPS, APS, RAPS)

In [ ]:
from src.methods.scores import (
    get_HPS_scores, get_APS_scores, get_RAPS_scores,
    get_APS_scores_all, get_RAPS_scores_all
)
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Create synthetic softmax probabilities (n=200, K=10 classes)
torch.manual_seed(42)
logits = torch.randn(200, 10, device=device)
probs = torch.softmax(logits, dim=1)
labels = torch.randint(0, 10, (200,), device=device)

# --- HPS scores ---
hps = get_HPS_scores(probs, labels)
assert hps.shape == (200,), f"HPS shape wrong: {hps.shape}"
assert (hps >= 0).all() and (hps <= 1).all(), "HPS scores out of [0,1]"
# HPS = 1 - p(true class), so higher prob => lower score
print(f"HPS: shape={hps.shape}, range=[{hps.min():.4f}, {hps.max():.4f}], mean={hps.mean():.4f}")

# --- APS scores ---
aps = get_APS_scores(probs, labels, randomize=True, seed=0)
assert aps.shape == (200,), f"APS shape wrong: {aps.shape}"
assert (aps >= 0).all() and (aps <= 1).all(), f"APS scores out of [0,1]: min={aps.min()}, max={aps.max()}"
print(f"APS: shape={aps.shape}, range=[{aps.min():.4f}, {aps.max():.4f}], mean={aps.mean():.4f}")

# --- RAPS scores ---
raps = get_RAPS_scores(probs, labels, lmbda=0.01, kreg=5, randomize=True, seed=0)
assert raps.shape == (200,), f"RAPS shape wrong: {raps.shape}"
print(f"RAPS: shape={raps.shape}, range=[{raps.min():.4f}, {raps.max():.4f}], mean={raps.mean():.4f}")

# --- APS all-class scores ---
aps_all = get_APS_scores_all(probs, randomize=True, seed=0)
assert aps_all.shape == (200, 10), f"APS_all shape wrong: {aps_all.shape}"
print(f"APS_all: shape={aps_all.shape}")

# --- RAPS all-class scores ---
raps_all = get_RAPS_scores_all(probs, lmbda=0.01, kreg=5, randomize=True, seed=0)
assert raps_all.shape == (200, 10), f"RAPS_all shape wrong: {raps_all.shape}"
print(f"RAPS_all: shape={raps_all.shape}")

# Verify APS true-class scores match APS_all indexed at true labels
aps_from_all = aps_all[torch.arange(200), labels]
diff = (aps - aps_from_all).abs().max()
print(f"APS consistency check (max diff): {diff:.6f}")
assert diff < 1e-4, f"APS scores inconsistent: {diff}"

print("[PASS] All scoring functions OK")

## 11. Functional Tests — Conformal Calibration & Coverage

In [ ]:
from src.methods.scores import get_conformal_quantile, get_HPS_scores, get_APS_scores
import numpy as np
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
alpha = 0.1  # target miscoverage rate

# Simulate a well-calibrated model: softmax probs for 500 cal + 500 test
torch.manual_seed(123)
n_cal, n_test, K = 500, 500, 10
logits_all = torch.randn(n_cal + n_test, K, device=device)
probs_all = torch.softmax(logits_all, dim=1)
labels_all = torch.randint(0, K, (n_cal + n_test,), device=device)

probs_cal, probs_test = probs_all[:n_cal], probs_all[n_cal:]
labels_cal, labels_test = labels_all[:n_cal], labels_all[n_cal:]

# --- HPS calibration ---
cal_scores_hps = get_HPS_scores(probs_cal, labels_cal).detach().cpu().numpy()
qhat_hps = get_conformal_quantile(cal_scores_hps, alpha=alpha, default_qhat=np.inf, exact_coverage=False)
print(f"HPS qhat: {qhat_hps:.4f}")

# Check coverage on test set
test_scores_hps = get_HPS_scores(probs_test, labels_test).detach().cpu().numpy()
coverage_hps = np.mean(test_scores_hps <= qhat_hps)
print(f"HPS coverage: {coverage_hps:.3f} (target >= {1 - alpha})")

# Build HPS prediction sets and check sizes
hps_all_test = (1 - probs_test).detach().cpu().numpy()  # scores for all classes
pred_sets_hps = (hps_all_test <= qhat_hps).astype(int)
avg_size_hps = pred_sets_hps.sum(axis=1).mean()
print(f"HPS avg prediction set size: {avg_size_hps:.2f}")

# --- APS calibration ---
cal_scores_aps = get_APS_scores(probs_cal, labels_cal, randomize=False).detach().cpu().numpy()
qhat_aps = get_conformal_quantile(cal_scores_aps, alpha=alpha, default_qhat=np.inf, exact_coverage=False)
print(f"\nAPS qhat: {qhat_aps:.4f}")

test_scores_aps = get_APS_scores(probs_test, labels_test, randomize=False).detach().cpu().numpy()
coverage_aps = np.mean(test_scores_aps <= qhat_aps)
print(f"APS coverage: {coverage_aps:.3f} (target >= {1 - alpha})")

# Sanity: qhat should be finite
assert np.isfinite(qhat_hps), "qhat_hps is not finite!"
assert np.isfinite(qhat_aps), "qhat_aps is not finite!"

print("[PASS] Conformal calibration & coverage OK")

## 12. Functional Tests — Differentiable Scoring & Gradient Flow

In [ ]:
from src.methods.scores import compute_scores_diff, compute_scores_diff_RAPS
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Create probabilities that require gradients
torch.manual_seed(77)
logits = torch.randn(16, 10, device=device, requires_grad=True)
probs = torch.softmax(logits, dim=1)
labels = torch.randint(0, 10, (16,), device=device)

# --- compute_scores_diff ---
scores = compute_scores_diff(probs, labels, device)
assert scores.shape == (16,), f"scores shape wrong: {scores.shape}"
print(f"Diff APS scores: shape={scores.shape}, mean={scores.mean().item():.4f}")

# Check gradient flow
loss = scores.sum()
loss.backward()
assert logits.grad is not None, "No gradients on logits!"
assert logits.grad.abs().sum() > 0, "Gradients are all zero!"
print(f"Gradient norm: {logits.grad.norm().item():.4f}")
print("[PASS] compute_scores_diff gradient flow OK")

# --- compute_scores_diff_RAPS ---
logits2 = torch.randn(16, 10, device=device, requires_grad=True)
probs2 = torch.softmax(logits2, dim=1)
scores_raps = compute_scores_diff_RAPS(probs2, labels, device, lambda_RAPS=0.01, k_RAPS=5.0)
assert scores_raps.shape == (16,), f"RAPS scores shape wrong: {scores_raps.shape}"

loss2 = scores_raps.sum()
loss2.backward()
assert logits2.grad is not None, "No gradients on logits2!"
print(f"RAPS gradient norm: {logits2.grad.norm().item():.4f}")
print("[PASS] compute_scores_diff_RAPS gradient flow OK")

## 13. Functional Tests — Smooth Quantile

In [ ]:
from src.methods.scores import Smoothquantile
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Test with known data: sorted scores 0.01, 0.02, ..., 1.00
scores = torch.arange(1, 101, dtype=torch.float32, device=device) / 100.0
alpha = 0.1

q = Smoothquantile(scores, alpha, device)
print(f"Smooth quantile (alpha={alpha}): {q.item():.4f}")
# For 100 uniformly spaced scores, the 90th percentile should be ~0.90
assert 0.85 < q.item() < 0.95, f"Smooth quantile seems off: {q.item()}"

# Test that it's differentiable
scores_grad = torch.randn(50, device=device, requires_grad=True)
q_grad = Smoothquantile(scores_grad, 0.1, device)
q_grad.backward()
assert scores_grad.grad is not None, "Smoothquantile not differentiable!"
print(f"Smoothquantile gradient norm: {scores_grad.grad.norm().item():.4f}")

print("[PASS] Smoothquantile OK")

## 14. Functional Tests — Size Loss Estimation

In [ ]:
from src.methods.scores import Estimate_size_loss_HPS_hard, Estimate_size_loss_HPS
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(99)
n, K = 32, 10
probs = torch.softmax(torch.randn(n, K, device=device), dim=1)
labels = torch.randint(0, K, (n,), device=device)
tau = torch.tensor([0.8], device=device)

# --- Estimate_size_loss_HPS ---
size_loss, class_loss = Estimate_size_loss_HPS(probs, labels, tau, device, num_classes=K, T=1.0, K=0.0)
print(f"HPS size_loss: {size_loss.item():.4f}, class_loss: {class_loss.item():.4f}")
assert size_loss.item() >= 0, "size_loss should be non-negative"

# --- Estimate_size_loss_HPS_hard ---
soft_size, hard_size, class_loss_h = Estimate_size_loss_HPS_hard(probs, labels, tau, device, num_classes=K, T=1.0, K=0.0)
print(f"HPS_hard: soft_size={soft_size.item():.4f}, hard_size={hard_size.item():.4f}, class_loss={class_loss_h.item():.4f}")
assert hard_size.item() >= 1.0, "hard_size should be >= 1 (at least true class)"
print(f"Avg hard prediction set size: {hard_size.item():.2f}")

print("[PASS] Size loss estimation OK")

## 15. Functional Tests — End-to-End Conformal Pipeline

In [ ]:
import torch
import numpy as np
from src.models.resnet import resnet
from src.methods.scores import get_HPS_scores, get_APS_scores, get_conformal_quantile

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. Create a model
model = resnet(depth=20, num_classes=10, use_fc_single=False).to(device)
model.eval()

# 2. Generate synthetic data (n=300 examples, 3x32x32 images, 10 classes)
torch.manual_seed(2024)
n_total = 300
X = torch.randn(n_total, 3, 32, 32, device=device)
Y = torch.randint(0, 10, (n_total,), device=device)

# Split: 200 cal, 100 test
n_cal = 200
X_cal, X_test = X[:n_cal], X[n_cal:]
Y_cal, Y_test = Y[:n_cal], Y[n_cal:]

# 3. Get model predictions
with torch.no_grad():
    probs_cal = torch.softmax(model(X_cal), dim=1)
    probs_test = torch.softmax(model(X_test), dim=1)

# 4. Calibrate with HPS
alpha = 0.1
cal_scores = get_HPS_scores(probs_cal, Y_cal).detach().cpu().numpy()
qhat = get_conformal_quantile(cal_scores, alpha=alpha, default_qhat=np.inf, exact_coverage=False)
print(f"Step 1 - Calibration: qhat = {qhat:.4f}")

# 5. Form prediction sets on test data
test_hps_all = (1 - probs_test).detach().cpu().numpy()  # HPS scores for all classes
pred_sets = (test_hps_all <= qhat)  # Boolean mask: which classes are in the set

# 6. Check marginal coverage
test_scores = get_HPS_scores(probs_test, Y_test).detach().cpu().numpy()
coverage = np.mean(test_scores <= qhat)
avg_size = pred_sets.sum(axis=1).mean()
print(f"Step 2 - Coverage: {coverage:.3f} (target >= {1-alpha:.1f})")
print(f"Step 3 - Avg set size: {avg_size:.2f} / 10 classes")

# 7. Verify prediction set contains true label
true_in_set = pred_sets[np.arange(100), Y_test.cpu().numpy()]
coverage_check = true_in_set.mean()
print(f"Step 4 - True label in set: {coverage_check:.3f}")

# 8. Repeat with APS
cal_scores_aps = get_APS_scores(probs_cal, Y_cal, randomize=False).detach().cpu().numpy()
qhat_aps = get_conformal_quantile(cal_scores_aps, alpha=alpha, default_qhat=np.inf, exact_coverage=False)
test_scores_aps = get_APS_scores(probs_test, Y_test, randomize=False).detach().cpu().numpy()
coverage_aps = np.mean(test_scores_aps <= qhat_aps)
print(f"\nAPS - qhat={qhat_aps:.4f}, coverage={coverage_aps:.3f}")

print("\n[PASS] End-to-end conformal pipeline OK")

## 16. Functional Tests — Smooth APS Scores (JAX)

In [ ]:
from src.methods.smooth_conformal import smooth_aps_score, smooth_aps_score_all
from src.methods.scores import get_sos
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Small example: n=8 examples, K=5 classes (sos needs matching comm pattern)
torch.manual_seed(55)
K = 5
n = 8
logits = torch.randn(n, K, device=device)
probs = torch.softmax(logits, dim=1)
labels = torch.randint(0, K, (n,), device=device)

# Set up smooth order stat object
sos = get_sos(K)
print(f"VariationalSortingNet for K={K} classes initialized")

# --- smooth_aps_score (scores for true labels) ---
scores = smooth_aps_score(probs, labels, sos=sos, device=device, dispersion=0.1, rng=None)
assert scores.shape == (n,), f"smooth_aps_score shape wrong: {scores.shape}"
print(f"smooth_aps_score: shape={scores.shape}, values={scores.detach().cpu().numpy()}")
assert (scores >= 0).all(), "Scores should be non-negative"

# --- smooth_aps_score_all (scores for all classes) ---
scores_all = smooth_aps_score_all(probs, sos=sos, device=device, dispersion=0.1, rng=None)
assert scores_all.shape == (n, K), f"smooth_aps_score_all shape wrong: {scores_all.shape}"
print(f"smooth_aps_score_all: shape={scores_all.shape}")

# Consistency: scores at true labels should match smooth_aps_score
scores_at_true = scores_all[torch.arange(n), labels.cpu()]
diff = (scores.cpu() - scores_at_true).abs().max()
print(f"Consistency check (max diff): {diff:.6f}")

print("[PASS] Smooth APS scores (JAX) OK")

## 17. Smoke Test — scripts/train.py argparse

In [ ]:
!python scripts/train.py --help 2>&1 | head -20
print("\n[PASS] scripts/train.py --help OK")

## Summary

In [ ]:
print("="*50)
print("All import, smoke, and functional tests passed!")
print("The restructured repo works correctly.")
print("="*50)